In [1]:
import sys
from pathlib import Path
import cProfile
import pstats
from time import perf_counter
from dataclasses import asdict


import pandas as pd
from datetime import datetime, timezone
import json
from pickle import load
from random import randrange
from datetime import timedelta

from sklearn.pipeline import Pipeline
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

from lightgbm import LGBMClassifier
from onnxmltools.convert.lightgbm.operator_converters.LightGbm import convert_lightgbm
from skl2onnx.common.shape_calculator import calculate_linear_classifier_output_shapes
from skl2onnx import update_registered_converter

# Racine du projet : OC-Projet-8/ (2 niveaux au-dessus de src/notebooks/)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks" and ROOT.parent.name == "src":
    ROOT = ROOT.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

#BASE_DIR = Path(__file__).resolve().parents[2]
DATA_PATH = ROOT / "data" / "original" / "demonstration_data.csv"
SCHEMA_PATH = ROOT / "data" / "schema" / "typeAdapters.json"
MODEL_PATH = ROOT / "src" / "model" / "lgb_model.pkl"
PARAMS_PATH = ROOT / "data" / "profils.json"

In [2]:
from src.database.database import engine
from src.database.simulate_client import (
    random_datetime_between, 
    simulate_profils
)

from src.monitoring.data import (
    compute_kpis,
    default_date_range,
    load_logs,
)

from src.database.database import save_error_log, save_prediction_log
from src.utils.utils import custom_sampler_ratio, business_cost
from src.api import scoring_api

In [3]:
start, end = default_date_range()

## Latences enregistrées en base

In [ ]:
full_data = load_logs(start, end)

### Volume et latences globales

In [4]:
kpis = compute_kpis(full_data)
pd.Series(kpis)

total         6565.000000
success       5414.000000
errors        1151.000000
error_rate       0.175324
p50_ms          13.645000
p95_ms          68.300400
dtype: float64

- total: nombre total de requêtes
- success: nombre de requêtes correctes ayant abouties à une prédiction
- errors: nombre de requêtes erronées ayant été rejetées par l'API (inputs incorrects)
- p50_ms: latence globale médiane
- p95_ms: latence globale des 5% de requêtes les plus longues

In [5]:
success = full_data.loc[~full_data["error"]]
errors = full_data.loc[full_data["error"]]

### Latence — requêtes réussies

`execution_time_ms` mesure le **temps total** côté API : validation + inférence + écriture PostgreSQL.

In [6]:
success.describe().execution_time_ms

count    5414.000000
mean       30.135364
std        99.541954
min        10.751000
25%        12.846000
50%        14.351500
75%        35.406000
max      2483.957000
Name: execution_time_ms, dtype: float64

In [7]:
outliers = (len(success.loc[success["execution_time_ms"] > 100])/3901)*100
print(f"Pourcentage de requêtes longues (>100ms) : {outliers:.2f}%")

Pourcentage de requêtes longues (>100ms) : 0.79%


Médiane à 13ms, moyenne à 21ms. La moyenne semble tirée vers le haut par un petit nombre de valeurs extrêmes.

### Latence — requêtes échouées

In [8]:
errors.describe().execution_time_ms

count    1151.000000
mean        1.130141
std         1.247760
min         0.000000
25%         0.000000
50%         1.545000
75%         2.046000
max        11.649000
Name: execution_time_ms, dtype: float64

La latence des erreurs dépend de l'étape du pipeline ayant rejeté l'input et donc du type d'erreur.

In [9]:
errors.error_message.unique()

array(['No features value was recieved from the user.',
       'Unknown feature, not suported by pydantic validation : invalid_feature',
       'Erreurs de validation :\n\n- - `EXT_SOURCE_1` : Input should be a valid number, unable to parse string as a number — valeur reçue : `pas un nombre`',
       'Erreurs de validation :\n\n- - `DAYS_BIRTH` : Input should be a valid integer, unable to parse string as an integer — valeur reçue : `date inconnue`',
       'Erreurs de validation :\n\n- - `AMT_CREDIT` : Input should be a valid number, unable to parse string as a number — valeur reçue : ``',
       'Aucune valeur utilisateur reçue.',
       'Variable inconnue, non prise en charge dans la validation pydantic : FEATURE_INEXISTANTE',
       'Variable inconnue, non prise en charge dans la validation pydantic : invalid_feature'],
      dtype=object)

In [10]:
for error in errors.error_message.unique():
    print(error)
    print(errors.loc[errors["error_message"] == error].describe().execution_time_ms)
    print("\n")


No features value was recieved from the user.
count    557.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: execution_time_ms, dtype: float64


Unknown feature, not suported by pydantic validation : invalid_feature
count    585.000000
mean       2.175971
std        0.752917
min        1.353000
25%        1.801000
50%        2.029000
75%        2.271000
max       11.649000
Name: execution_time_ms, dtype: float64


Erreurs de validation :

- - `EXT_SOURCE_1` : Input should be a valid number, unable to parse string as a number — valeur reçue : `pas un nombre`
count    3.000000
mean     5.181667
std      3.460523
min      1.986000
25%      3.344000
50%      4.702000
75%      6.779500
max      8.857000
Name: execution_time_ms, dtype: float64


Erreurs de validation :

- - `DAYS_BIRTH` : Input should be a valid integer, unable to parse string as an integer — valeur reçue : `date inconnue`
count    1.000
mean     2.079
std       

On observe que deux messages d'erreurs sont majoritairement représentés dans la base: aucune feature reçue et feature inconnue. La première entraine un rejet de l'input avant tout traitement et renvoie donc une latence égale à 0ms. La seconde montre une moyenne et une médiane à 2ms. Quelques latence plus importantes existent toutefois, avec un maximum à 11ms.

**Interprétation :** Les succés (~13 ms P50) sont beaucoup plus coûteux (**~10× plus**) que le chemin d'erreur (~1,5 ms P50). L'inférence via la fonction infer_from_new_vector explique donc probablement le temps de réponse plus important dans le cas d'une requête valide.

## Profiling cProfile

In [4]:
N_FEATURES = 262 #Nb de features attendues par le model
N_PROFILE_CALLS = 500
EVENT_TIMES = [random_datetime_between(start, end).isoformat() for ndate in range(N_PROFILE_CALLS)]
SIMULATE_PROFILS = False

In [5]:
#Tableau de données de démonstration
data = pd.read_csv(DATA_PATH, sep = ";")

#Modèle de scoring
with open(MODEL_PATH, "rb") as f:
    scoring_model = load(f)

#Dictionnaire de nouveaux paramètres

if SIMULATE_PROFILS:
    simulate_profils(ROOT, N_PROFILE_CALLS, 0) #On simule 500 nouveaux clients corrects 

with open(PARAMS_PATH) as f:
    params_list = json.load(f)


In [6]:
update_registered_converter(
    LGBMClassifier, 
    "LightGbmLGBMClassifier",
    calculate_linear_classifier_output_shapes, 
    convert_lightgbm,
    options={"nocl": [True, False], "zipmap": [True, False, "columns"]}
)

inference_steps = [
    ('imputer', scoring_model.estimator.named_steps['simpleimputer']), 
    ('transformer', scoring_model.estimator.named_steps['powertransformer']),
    ('scaler', scoring_model.estimator.named_steps['minmaxscaler']), 
    ('classifier', scoring_model.estimator.named_steps['lgbmclassifier']),
]

sklearn_pipe = Pipeline(inference_steps)

# 3. Définir le type d'entrée et convertir
initial_type = [('float_input', FloatTensorType([None, N_FEATURES]))]

target_opsets = {
    '': 15,          # Domaine par défaut (ai.onnx)
    'ai.onnx.ml': 3  # Domaine Machine Learning (LightGBM, Arbres, etc.)
}

onnx_model = convert_sklearn(sklearn_pipe, initial_types=initial_type, target_opset=target_opsets)

In [7]:
SAVING_DIR = ROOT / "src" / "model"
with open(SAVING_DIR / "onnx_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

In [20]:
#MODEL = scoring_model
MODEL = SAVING_DIR / "onnx_model.onnx"

In [21]:
def run_validate_params(params: dict):
    params_df = pd.DataFrame(
        {
            "feature": list(params.keys()),
            "value": list(params.values()),
        }
    )
    
    scoring_api.validate_params(params_df, scoring_api.explicative_features)

def run_inference(params: dict, event_time: datetime):
    scoring_api.infer_from_new_vector(
        params = params,
        model = MODEL,
        start_time=perf_counter(),
        event_time = event_time,
        persist = False,
        onnx = True
        )

def run_pipeline(params: dict, event_time: str):
    scoring_api.process_scoring_request(
        user_values = params,
        model = MODEL,
        simulated_event_time = event_time,
        persist = True,
        onnx = True
        )

In [22]:
scenarios = [
    {
        "key":"A",
        "title":"Validation Pydantic",
        "description":"`validate_params` seul (sans inférence ni DB).",
        "function": run_validate_params,
    },
    {
        "key":"B",
        "title":"Inférence sans DB",
        "description":"`infer_from_new_vector` sans écriture PostgreSQL.",
        "function":run_inference,
    },
    {
        "key":"C",
        "title":"Chemin API complet",
        "description":"`process_scoring_request` (validation + inférence + DB).",
        "function":run_pipeline,
    },
]

### Profilage modèle sklearn

In [14]:
profilers = {}
for s in scenarios:
    profiler = cProfile.Profile()
    profiler.enable()
    for index in range(N_PROFILE_CALLS):
        if s['key'] == 'A':
            s['function'](params_list[index])
        elif s['key'] == 'B':
            s['function'](params_list[index], datetime.fromisoformat(EVENT_TIMES[index]))
        else:
            s['function'](params_list[index], EVENT_TIMES[index])
    profiler.disable()
    profilers[s['key']] = profiler

c:\Users\Iwiko\Documents\Travail\Data_Science\Formation\OC\Projet_8_nouveau\OC-Projet-8\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Iwiko\Documents\Travail\Data_Science\Formation\OC\Projet_8_nouveau\OC-Projet-8\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Iwiko\Documents\Travail\Data_Science\Formation\OC\Projet_8_nouveau\OC-Projet-8\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Iwiko\Documents\Travail\Data_Science\Formation\OC\Projet_8_nouveau\OC-Projet-8\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier

In [15]:
RESULT_COLUMNS = [
    "scenario",
    "filename",
    "func",
    "ncalls",
    "tottime",
    "percall_tottime",
    "cumtime",
]

scenario_titles = {s["key"]: s["title"] for s in scenarios}

frames = []
avg_ms = []
for index, key in enumerate(profilers):
    p = pstats.Stats(profilers[key]).sort_stats('cumulative')
    p.dump_stats(ROOT / "data" / f'profile_{key}.pstats')
    avg_ms.append(1000*(p.total_tt/N_PROFILE_CALLS))
    print(f"Durée moyenne (ms) - {scenario_titles[key]} : {avg_ms[index]}")

    df = pd.DataFrame(
        [
            {"func": func, **asdict(stats)}
            for func, stats in p.get_stats_profile().func_profiles.items()
        ]
    )
    df = df[~df["func"].str.contains("run", na=False)] #exclue les wrappers du notebook
    #top = df.sort_values("cumtime", ascending=False).head(15).copy()
    top = df.head(15).copy()
    top["scenario"] = scenario_titles[key]
    top["filename"] = top["file_name"].apply(lambda path: Path(path).name)
    frames.append(top[RESULT_COLUMNS])

profile_results = pd.concat(frames, ignore_index=True)

Durée moyenne (ms) - Validation Pydantic : 2.8708178000000006
Durée moyenne (ms) - Inférence sans DB : 25.57492759999999
Durée moyenne (ms) - Chemin API complet : 31.775827999999994


In [16]:
print(f"La validation représente {round((avg_ms[0]/avg_ms[2])*100)}% du temps complet de traitement")
print(f"L'inférence représente {round(((avg_ms[1]-avg_ms[0])/avg_ms[2])*100)}% du temps complet de traitement avec une durée moyenne de {round(avg_ms[1] - avg_ms[0])}ms")
print(f"L'enregistrement en base de données représente {round(((avg_ms[2]-avg_ms[1])/avg_ms[2])*100)}% du temps complet de traitement avec une durée moyenne de {round(avg_ms[2]- avg_ms[1])}ms")

La validation représente 9% du temps complet de traitement
L'inférence représente 71% du temps complet de traitement avec une durée moyenne de 23ms
L'enregistrement en base de données représente 20% du temps complet de traitement avec une durée moyenne de 6ms


In [17]:
profile_results.loc[profile_results.scenario == "Validation Pydantic"].iloc[0:4]

,scenario,filename,func,ncalls,tottime,percall_tottime,cumtime
0,Validation Pydantic,scoring_api.py,validate_params,500,0.024,0.0,1.268
1,Validation Pydantic,managers.py,__init__,500,0.000,0.0,0.000
2,Validation Pydantic,frame.py,iterrows,4893,0.018,0.0,0.490
3,Validation Pydantic,type_adapter.py,_init_core_attrs,4393,0.045,0.0,0.426


Le coût en temps de traitement lors de la validation des paramètres est dominé par la création répétée de `TypeAdapter` et `DataFrame.iterrows` dans `validate_params` (boucle for par ligne du dictionnaire params).

In [18]:
profile_results.loc[profile_results.scenario == 'Inférence sans DB']

,scenario,filename,func,ncalls,tottime,percall_tottime,cumtime
15,Inférence sans DB,scoring_api.py,infer_from_new_vector,500,0.014,0.0,12.879
16,Inférence sans DB,basic.py,predict,500,0.007,0.0,0.147
17,Inférence sans DB,_response.py,_get_response_values_binary,500,0.004,0.0,11.718
18,Inférence sans DB,_response.py,_get_response_values,500,0.010,0.0,11.383
19,Inférence sans DB,sklearn.py,predict_proba,500,0.013,0.0,1.711
20,Inférence sans DB,_set_output.py,wrapped,2000/1500,0.011,0.0,9.016
21,Inférence sans DB,_data.py,transform,500,0.010,0.0,0.130
22,Inférence sans DB,validation.py,validate_data,2500,0.032,0.0,4.848
23,Inférence sans DB,validation.py,check_array,3500,0.170,0.0,4.100
24,Inférence sans DB,_base.py,_validate_input,500,0.008,0.0,3.895


En ajoutant l'inférence, ce sont les fonction sklearn/imbalance-learn qui apparaissent les plus couteuses.

In [19]:
profile_results.loc[profile_results.scenario == 'Chemin API complet'].iloc[0:10]

,scenario,filename,func,ncalls,tottime,percall_tottime,cumtime
30,Chemin API complet,scoring_api.py,process_scoring_request,500,0.022,0.0,15.987
31,Chemin API complet,scoring_api.py,infer_from_new_vector,500,0.027,0.0,13.725
32,Chemin API complet,basic.py,predict,500,0.007,0.0,0.160
33,Chemin API complet,_response.py,_get_response_values_binary,500,0.005,0.0,11.423
34,Chemin API complet,_response.py,_get_response_values,500,0.009,0.0,11.089
35,Chemin API complet,sklearn.py,predict_proba,500,0.013,0.0,1.703
36,Chemin API complet,_set_output.py,wrapped,2000/1500,0.011,0.0,8.748
37,Chemin API complet,_data.py,transform,500,0.010,0.0,0.128
38,Chemin API complet,validation.py,validate_data,2500,0.030,0.0,4.677
39,Chemin API complet,validation.py,check_array,3500,0.164,0.0,3.948


La même conclusion qu'en scénario B apparait sur le pipeline complet.

### Profilage model onnx

In [23]:
profilers_onnx = {}
for s in scenarios:
    profiler = cProfile.Profile()
    profiler.enable()
    for index in range(N_PROFILE_CALLS):
        if s['key'] == 'A':
            s['function'](params_list[index])
        elif s['key'] == 'B':
            s['function'](params_list[index], datetime.fromisoformat(EVENT_TIMES[index]))
        else:
            s['function'](params_list[index], EVENT_TIMES[index])
    profiler.disable()
    profilers_onnx[s['key']] = profiler

In [24]:
RESULT_COLUMNS = [
    "scenario",
    "filename",
    "func",
    "ncalls",
    "tottime",
    "percall_tottime",
    "cumtime",
]

scenario_titles = {s["key"]: s["title"] for s in scenarios}

frames_onnx = []
avg_ms_onnx = []
for index, key in enumerate(profilers_onnx):
    p = pstats.Stats(profilers_onnx[key]).sort_stats('cumulative')
    p.dump_stats(ROOT / "data" / f'profile_{key}_onnx.pstats')
    avg_ms_onnx.append(1000*(p.total_tt/N_PROFILE_CALLS))
    print(f"Durée moyenne (ms) - {scenario_titles[key]} : {avg_ms_onnx[index]}")

    df = pd.DataFrame(
        [
            {"func": func, **asdict(stats)}
            for func, stats in p.get_stats_profile().func_profiles.items()
        ]
    )
    df = df[~df["func"].str.contains("run", na=False)] #exclue les wrappers du notebook
    #top = df.sort_values("cumtime", ascending=False).head(15).copy()
    top = df.head(15).copy()
    top["scenario"] = scenario_titles[key]
    top["filename"] = top["file_name"].apply(lambda path: Path(path).name)
    frames_onnx.append(top[RESULT_COLUMNS])

profile_results_onnx = pd.concat(frames_onnx, ignore_index=True)

Durée moyenne (ms) - Validation Pydantic : 3.0242446000000007
Durée moyenne (ms) - Inférence sans DB : 15.049619599999998
Durée moyenne (ms) - Chemin API complet : 18.23544160000002


In [25]:
print(f"La validation représente {round((avg_ms_onnx[0]/avg_ms_onnx[2])*100)}% du temps complet de traitement")
print(f"L'inférence représente {round(((avg_ms_onnx[1]-avg_ms_onnx[0])/avg_ms_onnx[2])*100)}% du temps complet de traitement avec une durée moyenne de {round(avg_ms_onnx[1] - avg_ms_onnx[0])}ms")
print(f"L'enregistrement en base de données représente {round(((avg_ms_onnx[2]-avg_ms_onnx[1])/avg_ms_onnx[2])*100)}% du temps complet de traitement avec une durée moyenne de {round(avg_ms_onnx[2]- avg_ms_onnx[1])}ms")

La validation représente 17% du temps complet de traitement
L'inférence représente 66% du temps complet de traitement avec une durée moyenne de 12ms
L'enregistrement en base de données représente 17% du temps complet de traitement avec une durée moyenne de 3ms
